# Vocabulary Level Prediction — RoBERTa comparison

Fine-tune `roberta-base` on **one consensus target** (`target_vocab` = mean of raters, same as `Model_selection`). Development split on filtered train; final fit on **full** train; evaluation on **test** (same metrics/plots as §6 there). **Weighted MSE** on train matches rare-class emphasis from classical CV.

## 0. Environment

`importlib.reload` on `utils`, `modeling_utils`, `transformers_utils` after editing `.py` files (same pattern as [Model_selection.ipynb](Model_selection.ipynb)).

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments

import modeling_utils
import transformers_utils
import utils

importlib.reload(utils)
importlib.reload(modeling_utils)
importlib.reload(transformers_utils)

from modeling_utils import (
    filter_by_rater_agreement,
    create_consensus_target,
    build_text_target_dataset,
    build_stratification_bins,
    evaluate_fold,
    round_and_clip_predictions,
    plot_ordinal_predicted_vs_true_scatter,
    plot_ordinal_continuous_boxplot_by_true_class,
    plot_row_normalized_confusion_matrix_from_labels,
)
from transformers_utils import (
    WeightedRegressionTrainer,
    build_hf_regression_compute_metrics_fn,
    build_tokenized_regression_dataset,
    build_weighted_regression_data_collator,
    prepare_balanced_regression_weights,
)

pd.set_option("display.max_colwidth", None)

## 1. Load data

`Data/train.csv`, `Data/test.csv` (same 80/20 split as EDA).

In [ ]:
df_train_raw = pd.read_csv("Data/train.csv")
df_test_raw = pd.read_csv("Data/test.csv")
print(f"Train rows: {len(df_train_raw)}  |  Test rows: {len(df_test_raw)}")

## 2. Preprocessing

**Train:** agreement filter + consensus `target_vocab` (same as model selection).  
**Test:** no agreement filter; consensus label for evaluation only (deployment-like).

### 2.1 Train: filter and consensus target

In [ ]:
df_train = filter_by_rater_agreement(df_train_raw.copy())
df_train = create_consensus_target(df_train)
df_train_text = build_text_target_dataset(df_train)

### 2.2 Test: consensus label only (no row drops)

In [ ]:
df_test_eval = create_consensus_target(df_test_raw.copy())
df_test_text = build_text_target_dataset(df_test_eval)

## 3. Targets and stratification bins

Single scalar target `y`; bins for **stratified** train/val split only (`merge_below=2` aligned with `Model_selection`).

In [ ]:
CV_RANDOM_STATE = 42
VAL_FRACTION = 0.15

text_train_all = df_train_text["Text_cleaned"].reset_index(drop=True)
y_train_all = np.asarray(df_train_text["target_vocab"].values, dtype=float)
stratification_bins = build_stratification_bins(y_train_all.astype(int), merge_below=2)

## 4. Train / validation split (development)

**Decision:** 15% held out for early stopping / metric tracking; stratified on merged bins. **No test rows** used here.

In [ ]:
row_indices = np.arange(len(text_train_all))
idx_train, idx_val = train_test_split(
    row_indices,
    test_size=VAL_FRACTION,
    random_state=CV_RANDOM_STATE,
    stratify=stratification_bins,
)

text_tr = text_train_all.iloc[idx_train].tolist()
text_val = text_train_all.iloc[idx_val].tolist()
y_tr = y_train_all[idx_train]
y_val = y_train_all[idx_val]

sample_weight_tr = prepare_balanced_regression_weights(y_tr.astype(int))
# Validation loss: uniform weights (interpretable MSE); collator still requires the column.
sample_weight_val = np.ones(len(idx_val), dtype=np.float64)

print(f"Train (dev): {len(text_tr)}  |  Val: {len(text_val)}")

## 5. Model and tokenizer configuration

**Decision:** `roberta-base`, `AutoModelForSequenceClassification` with `num_labels=1`, `problem_type="regression"`. **Max length** capped for memory; increase if GPU allows.

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### 5.1 Tokenized datasets

HF ``datasets`` built in ``transformers_utils.build_tokenized_regression_dataset`` (same pattern as other notebooks: logic in utils, notebook calls only).

In [ ]:
ds_train_dev = build_tokenized_regression_dataset(
    tokenizer, text_tr, y_tr, sample_weight_tr, MAX_LENGTH
)
ds_val = build_tokenized_regression_dataset(
    tokenizer, text_val, y_val, sample_weight_val, MAX_LENGTH
)

## 6. Phase A — Fine-tune on train (dev split)

**Decision:** `WeightedRegressionTrainer` + balanced train weights; monitor **QWK** on val (same definition as `evaluate_fold`). Stop / pick best by val QWK.

In [ ]:
model_dev = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",
)

training_args_dev = TrainingArguments(
    output_dir="./roberta_dev_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    seed=CV_RANDOM_STATE,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = build_weighted_regression_data_collator(tokenizer)
compute_metrics = build_hf_regression_compute_metrics_fn(min_score=0, max_score=5)

trainer_dev = WeightedRegressionTrainer(
    model=model_dev,
    args=training_args_dev,
    train_dataset=ds_train_dev,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_dev.train()

## 7. Phase B — Refit on full train

**Decision:** Same hyperparameters as Phase A; **full** filtered train for final weights. **No** test data.

In [ ]:
sample_weight_full = prepare_balanced_regression_weights(y_train_all.astype(int))
text_full = text_train_all.tolist()
ds_train_full = build_tokenized_regression_dataset(
    tokenizer, text_full, y_train_all, sample_weight_full, MAX_LENGTH
)

model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",
)

training_args_full = TrainingArguments(
    output_dir="./roberta_full_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="no",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    seed=CV_RANDOM_STATE,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_full = WeightedRegressionTrainer(
    model=model_full,
    args=training_args_full,
    train_dataset=ds_train_full,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer_full.train()

## 8. Test-set evaluation

**Protocol:** `trainer_full` fit on full train only; test = **predict** + `evaluate_fold` + same ordinal plots as `Model_selection` §6. **No** test leakage.

### 8.1 Predict on test

In [ ]:
text_test = df_test_text["Text_cleaned"].reset_index(drop=True).tolist()
y_test = np.asarray(df_test_text["target_vocab"].values, dtype=int)
weights_test_dummy = np.ones(len(text_test), dtype=np.float32)

ds_test = build_tokenized_regression_dataset(
    tokenizer, text_test, y_test.astype(float), weights_test_dummy, MAX_LENGTH
)

pred_out = trainer_full.predict(ds_test)
y_pred_test = np.asarray(pred_out.predictions, dtype=float).squeeze()

### 8.2 Metrics (QWK, RMSE, MAE)

In [ ]:
test_metrics = evaluate_fold(y_test, y_pred_test, min_score=0, max_score=5)
summary_test = (
    pd.DataFrame([test_metrics], index=["Test set (RoBERTa)"])
    .rename(columns={"qwk": "QWK", "rmse": "RMSE", "mae": "MAE"})
)
display(summary_test)

### 8.3 Diagnostic plots (same as model selection §6)

In [ ]:
fig_scatter = plot_ordinal_predicted_vs_true_scatter(
    y_test,
    y_pred_test,
    title="Test set: predicted vs true (RoBERTa)",
    xlabel="True target_vocab",
)
plt.show()

fig_box = plot_ordinal_continuous_boxplot_by_true_class(
    y_test,
    y_pred_test,
    title="Test set: predicted score by true class (RoBERTa)",
    xlabel="True target_vocab",
)
plt.show()

y_pred_ord = round_and_clip_predictions(y_pred_test, 0, 5)
ordinal_labels = np.arange(0, 6)
fig_cm = plot_row_normalized_confusion_matrix_from_labels(
    y_test,
    y_pred_ord,
    labels=ordinal_labels,
    title="Test set: confusion matrix (row-normalized, RoBERTa)",
    xlabel="Predicted class (rounded & clipped)",
)
plt.show()